# Movie Discovery Assistant - GRPO Alignment Training

## Group Relative Policy Optimization (GRPO)
GRPO is the alignment method used by DeepSeek-R1. It's ideal for tasks where
quality is **relative** (recommendation A is better than B, but both are valid).

### Why GRPO for Movie Recommendations?
- **No reward model needed** (unlike PPO)
- **No preference pairs needed** (unlike DPO)
- **Self-improving**: Generates multiple responses, ranks them, learns from best
- **Rule-based rewards**: Define what makes a good recommendation programmatically
- **Best for reasoning**: Multi-criteria recommendations benefit from GRPO's approach

### How GRPO Works:
1. For each prompt, generate N responses (N=4)
2. Score each response with reward functions
3. Compute group-relative advantages (how much better than group average)
4. Update policy to favor higher-advantage responses

### Optimizations Applied:
- Flash Attention 2, Gradient Checkpointing, 8-bit AdamW
- Custom reward functions for movie recommendations
- KL penalty to prevent degeneration

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.5.0" xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

gpu_name = torch.cuda.get_device_name(0)
major_ver, _ = torch.cuda.get_device_capability()
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

dtype = torch.bfloat16 if major_ver >= 8 else torch.float16
max_seq_length = 2048

# Load base model (or SFT-trained model)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

# LoRA for GRPO
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)

print("Model loaded with LoRA for GRPO training")

In [ ]:
# ============================================================================
# Reward Functions for Movie Recommendations
# ============================================================================
# GRPO uses rule-based rewards instead of a learned reward model.
# Each function scores a response on a specific quality dimension.
#
# For movie recommendations, we care about:
# 1. FORMAT: Correct structure (numbered list, titles, details)
# 2. COUNT: Right number of recommendations (3-5)
# 3. DIVERSITY: Different genres/eras (not all from same year)
# 4. RELEVANCE: Matches what the user asked for
# ============================================================================
import re

def format_reward_func(completions, **kwargs):
    """Reward for correct response format."""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]["content"]
        score = 0.0

        # Has numbered recommendations
        numbered = re.findall(r'^\d+\.', text, re.MULTILINE)
        if len(numbered) >= 2:
            score += 0.3
        if len(numbered) >= 3:
            score += 0.2

        # Has bold titles
        if "**" in text:
            score += 0.2

        # Has years in parentheses
        years = re.findall(r'\(\d{4}\)', text)
        if years:
            score += 0.15

        # Has opening context line
        if text.strip() and not text.strip()[0].isdigit():
            score += 0.15

        rewards.append(min(score, 1.0))
    return rewards


def count_reward_func(completions, **kwargs):
    """Reward for having 3-5 recommendations."""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]["content"]
        numbered = re.findall(r'^\d+\.', text, re.MULTILINE)
        count = len(numbered)

        if 3 <= count <= 5:
            score = 1.0
        elif count == 2:
            score = 0.5
        elif count == 1:
            score = 0.2
        elif count > 5:
            score = 0.6
        else:
            score = 0.0
        rewards.append(score)
    return rewards


def diversity_reward_func(completions, **kwargs):
    """Reward for diverse recommendations."""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else completion[0]["content"]
        years = [int(y) for y in re.findall(r'\((\d{4})\)', text)]

        if len(years) < 2:
            rewards.append(0.3)
            continue

        # Year diversity
        year_range = max(years) - min(years)
        year_score = min(year_range / 30.0, 1.0)

        # Genre mention diversity
        genre_words = {"action", "comedy", "drama", "thriller", "horror", "sci-fi",
                       "romance", "adventure", "fantasy", "mystery", "animation"}
        found = sum(1 for g in genre_words if g in text.lower())
        genre_score = min(found / 3.0, 1.0)

        rewards.append(0.5 * year_score + 0.5 * genre_score)
    return rewards


def relevance_reward_func(completions, prompts=None, **kwargs):
    """Reward for matching query intent."""
    rewards = []
    for i, completion in enumerate(completions):
        text = completion if isinstance(completion, str) else completion[0]["content"]
        prompt = prompts[i] if prompts else ""
        prompt_lower = prompt.lower() if isinstance(prompt, str) else ""
        text_lower = text.lower()
        score = 0.0

        # Genre match
        genres = ["action", "comedy", "drama", "thriller", "horror", "sci-fi",
                  "romance", "adventure", "fantasy", "mystery"]
        for genre in genres:
            if genre in prompt_lower and genre in text_lower:
                score += 0.3
                break

        # Mood match
        moods = ["dark", "uplifting", "intense", "lighthearted", "emotional",
                 "mind-bending", "funny", "scary", "thought-provoking"]
        for mood in moods:
            if mood in prompt_lower and mood in text_lower:
                score += 0.3
                break

        # Length check (not too short, not too long)
        word_count = len(text.split())
        if 50 <= word_count <= 300:
            score += 0.2

        # Not repetitive
        sentences = text.split('.')
        unique_sentences = set(s.strip().lower() for s in sentences if len(s.strip()) > 10)
        if len(sentences) > 0 and len(unique_sentences) / max(len(sentences), 1) > 0.7:
            score += 0.2

        rewards.append(min(score, 1.0))
    return rewards


print("Reward functions defined:")
print("  1. format_reward_func   (weight: 0.2)")
print("  2. count_reward_func    (weight: 0.2)")
print("  3. diversity_reward_func(weight: 0.3)")
print("  4. relevance_reward_func(weight: 0.3)")

### Upload Dataset
Upload `train.jsonl` (same SFT data - GRPO uses the prompts only)

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload train.jsonl

In [ ]:
from datasets import load_dataset

# Load and prepare prompts for GRPO
# GRPO only needs prompts - it generates completions itself
raw_dataset = load_dataset("json", data_files="train.jsonl", split="train")

# Format prompts
def prepare_grpo_prompts(examples):
    prompts = []
    for instruction in examples["instruction"]:
        # Use chat format for GRPO
        prompts.append([
            {"role": "system", "content": "You are a helpful movie recommendation assistant. Provide detailed, diverse recommendations with titles, years, and descriptions."},
            {"role": "user", "content": instruction}
        ])
    return {"prompt": prompts}

grpo_dataset = raw_dataset.map(prepare_grpo_prompts, batched=True,
                                remove_columns=raw_dataset.column_names)

# Take a subset for GRPO (it's compute-intensive due to multiple generations)
grpo_dataset = grpo_dataset.shuffle(seed=3407).select(range(min(500, len(grpo_dataset))))

print(f"GRPO training prompts: {len(grpo_dataset)}")
print(f"Sample: {grpo_dataset[0]['prompt'][1]['content'][:100]}...")

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# ============================================================================
# GRPO Configuration
# ============================================================================
# GRPO generates num_generations responses per prompt, scores them,
# and updates the policy based on group-relative advantages.
#
# Key parameters:
# - num_generations: More = better advantage estimates, but slower
# - kl_coef: Prevents the model from deviating too far from base
# - temperature: Higher = more diverse generations for comparison
# ============================================================================

use_bf16 = torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    # --- GRPO Specific ---
    num_generations=4,                 # Generate 4 responses per prompt
    max_prompt_length=512,
    max_completion_length=512,
    temperature=0.9,                   # Diverse sampling

    # --- Training ---
    num_train_epochs=1,
    per_device_train_batch_size=1,     # Small batch (4 gens per sample)
    gradient_accumulation_steps=16,    # Effective batch = 16
    learning_rate=1e-5,                # Very low LR for RL
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",

    # --- Precision ---
    fp16=not use_bf16,
    bf16=use_bf16,

    # --- Logging ---
    logging_steps=1,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    output_dir="grpo_outputs",
    report_to="none",
    seed=3407,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        count_reward_func,
        diversity_reward_func,
        relevance_reward_func,
    ],
)

print("GRPO Trainer configured:")
print(f"  Generations per prompt: {grpo_config.num_generations}")
print(f"  Reward functions: 4 (format, count, diversity, relevance)")
print(f"  LR: {grpo_config.learning_rate}")
print(f"  Training prompts: {len(grpo_dataset)}")

In [ ]:
# ============================================================================
# Train GRPO
# ============================================================================
import time

start = time.time()
grpo_stats = grpo_trainer.train()
elapsed = time.time() - start

print(f"\n{'=' * 60}")
print(f"GRPO TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"  Time: {elapsed/60:.1f} minutes")
print(f"  Loss: {grpo_stats.training_loss:.4f}")
print(f"  Steps: {grpo_stats.global_step}")
print(f"{'=' * 60}")

In [ ]:
# ============================================================================
# Evaluate GRPO model
# ============================================================================
FastLanguageModel.for_inference(model)

test_queries = [
    "Recommend a mind-bending sci-fi movie",
    "I want a comedy from the 90s",
    "Dark thriller with a twist ending",
    "A lighthearted adventure for the family",
    "Movies directed by Christopher Nolan",
]

for query in test_queries:
    messages = [
        {"role": "system", "content": "You are a helpful movie recommendation assistant."},
        {"role": "user", "content": query},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
        )

    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    # Score with reward functions
    fmt_score = format_reward_func([response])[0]
    cnt_score = count_reward_func([response])[0]
    div_score = diversity_reward_func([response])[0]
    rel_score = relevance_reward_func([response], prompts=[query])[0]

    print(f"Q: {query}")
    print(f"A: {response[:300]}")
    print(f"Rewards: fmt={fmt_score:.2f} cnt={cnt_score:.2f} div={div_score:.2f} rel={rel_score:.2f}")
    print("-" * 60)

In [ ]:
# ============================================================================
# Save GRPO-aligned model
# ============================================================================
import shutil
from google.colab import files

model.save_pretrained("grpo_lora_model")
tokenizer.save_pretrained("grpo_lora_model")

# Export GGUF
print("Exporting to GGUF Q4_K_M...")
model.save_pretrained_gguf(
    "grpo_model_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

# Export merged 16-bit for vLLM
print("Exporting merged 16-bit for vLLM...")
model.save_pretrained_merged(
    "grpo_model_merged",
    tokenizer,
    save_method="merged_16bit",
)

shutil.make_archive('movie_assistant_grpo_lora', 'zip', 'grpo_lora_model')
shutil.make_archive('movie_assistant_grpo_gguf', 'zip', 'grpo_model_gguf')

files.download('movie_assistant_grpo_lora.zip')
files.download('movie_assistant_grpo_gguf.zip')
print("GRPO model saved and exported!")